# Actividad 3
__Curso:__ Tópicos avanzados en Inteligencia Artificial 1

__Programa:__ MIA 2-2025

__Profesor:__ Anthony D. Cho

__Ayudante corrector:__ Luis Oliveros

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaMunozS/topicos-avanzados/blob/main/notebooks/actividad3_Lopez_Munoz.ipynb)

**Guía de ejecución:** seleccionar GPU en Colab, ejecutar todas las celdas y autorizar Drive
si se desea conservar los pesos entre sesiones. La configuración completa usa imágenes de
512 × 512, un modelo de dos etapas y tres candidatos de tres etapas. Los resultados,
las respuestas numéricas y la justificación se calculan al ejecutar las preguntas.

**Estado de esta versión:** solución implementada para ejecutar el experimento completo.
Las pruebas técnicas previas no sustituyen el entrenamiento final ni sus resultados.
Antes de entregar en Webcursos, guardar este notebook con todas sus salidas de ejecución.

## Instrucciones
* La actividad debe ser realizada por los grupos capstones
* Por favor responder en este mismo notebook (una entrega por grupo)
* Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo actividad3_Sakuragi_Mitsui_Rukagua_Sendoh.ipynb
* Subir el archivo al link de entrega Actividad 3 en webcursos que será habilitado

__Fecha de entrega:__ Fecha límite de entrega 20 de septiembre de 2026 - 23:59 horas chile.

__Integrantes:__ (RUT, Nombre y Apellido)

* Ricardo Lopez
* Camilo Muñoz

**Para la entrega:** completar los RUT en la copia privada que se envíe a Webcursos.
La versión pública de GitHub identifica a los integrantes por sus nombres.

In [ ]:
# Configuración del experimento. Modificar aquí antes de ejecutar todo.
SEED = 84
IMG_SIZE = 512
BATCH_SIZE = 2                    # Conservador para una GPU T4.
MAX_EPOCHS = 80                   # Tope, no número de épocas obligatorio.
PATIENCE = 12
LEARNING_RATE = 1e-3
DROPOUT = 0.25
THRESHOLD = 0.5                   # Fijo para todos los modelos y particiones.
SAVE_TO_DRIVE = True              # En Colab solicita autorización una vez.
USE_MIXED_PRECISION = True        # Solo se activa si existe GPU.
REUSE_COMPLETED = True            # Reutiliza experimentos con idéntica configuración.
DATA_DIR_OVERRIDE = None          # Carpeta local ya descomprimida, si existe.
OUTPUT_DIR_OVERRIDE = None
Q2_CONFIGS = [
    {"stages": 3, "initial_filters": 15, "depth": 4},
    {"stages": 3, "initial_filters": 24, "depth": 4},
    {"stages": 3, "initial_filters": 16, "depth": 5},
]

## Librerias

In [ ]:
# Se aprovecha TensorFlow instalado en Colab, sin reemplazar su entorno GPU.
import importlib.util
import subprocess
import sys

dependencies = {
    "tensorflow": "tensorflow==2.18.1",
    "cv2": "opencv-python-headless==4.11.0.86",
    "sklearn": "scikit-learn>=1.4,<2",
    "pandas": "pandas>=2.2,<3",
    "matplotlib": "matplotlib>=3.8,<4",
}
missing = [package for module, package in dependencies.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Dependencias disponibles.")

In [ ]:
import gc
import hashlib
import json
import os
import platform
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import cv2
import numpy as np
import pandas as pd
from pandas import DataFrame
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, layers
from tensorflow.keras import backend as K

cv2.setNumThreads(1)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
PRECISION_POLICY = "mixed_float16" if gpus and USE_MIXED_PRECISION else "float32"
tf.keras.mixed_precision.set_global_policy(PRECISION_POLICY)
ENVIRONMENT = {
    "python": platform.python_version(), "tensorflow": tf.__version__,
    "keras": tf.keras.__version__, "numpy": np.__version__,
    "opencv": cv2.__version__, "pandas": pd.__version__,
    "gpu": [str(gpu) for gpu in gpus],
    "precision": tf.keras.mixed_precision.global_policy().name,
}
print(json.dumps(ENVIRONMENT, indent=2))
if not gpus:
    print("CPU detectada. El experimento completo a 512 px se recomienda en Colab con GPU.")

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if OUTPUT_DIR_OVERRIDE is not None:
    OUTPUT_ROOT = Path(OUTPUT_DIR_OVERRIDE)
elif IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/TopicosAvanzados/Tarea03")
else:
    OUTPUT_ROOT = Path.cwd() / "outputs" / "tarea03"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Resultados y pesos:", OUTPUT_ROOT)

## Funciones personalizadas

#### Graficador de desempeño

In [ ]:
def plot_history(history, width=12, height=10, save_path=None):
    """Empareja cada métrica con val_<métrica>, independientemente del orden."""
    values = history.history if hasattr(history, "history") else history
    keys = [key for key in ("loss", "iou", "dice") if key in values]
    fig, axes = plt.subplots(len(keys), 1, figsize=(width, height), squeeze=False)
    epochs = np.arange(1, len(values[keys[0]]) + 1)
    for ax, key in zip(axes.ravel(), keys):
        ax.plot(epochs, values[key], label="Entrenamiento")
        if "val_" + key in values:
            ax.plot(epochs, values["val_" + key], label="Validación")
        ax.set(xlabel="Época", ylabel=key, title=key.upper())
        ax.grid(alpha=0.25)
        ax.legend()
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)

#### [Dice Similarity Coefficient (DSC)](https://en.wikipedia.org/wiki/S%C3%B8rensen%E2%80%93Dice_coefficient)

In [ ]:
def dice_similarity_coef(y_true, y_pred, smooth=100):
    """Dice suave de la plantilla. No aplica umbral y conserva smooth=100."""
    truth = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    pred = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    return (2 * tf.reduce_sum(truth * pred) + smooth) / (
        tf.reduce_sum(truth) + tf.reduce_sum(pred) + smooth)


def dice_coef_loss(y_true, y_pred):
    return -dice_similarity_coef(y_true, y_pred)


def segmentation_loss(y_true, y_pred):
    """BCE + Dice suave por imagen, con el mismo peso en ambos modelos."""
    truth = tf.cast(y_true, tf.float32)
    pred = tf.cast(y_pred, tf.float32)
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(truth, pred))
    axes = (1, 2, 3)
    intersection = tf.reduce_sum(truth * pred, axis=axes)
    denominator = tf.reduce_sum(truth + pred, axis=axes)
    soft_dice = (2 * intersection + 1e-6) / (denominator + 1e-6)
    return 0.5 * bce + 0.5 * (1 - tf.reduce_mean(soft_dice))

#### [Intersection over union (IoU)](https://hasty.ai/content-hub/mp-wiki/metrics/iou-intersection-over-union)

In [ ]:
def iou(y_true, y_pred, smooth=100):
    """IoU suave de la plantilla, conservada para comparación explícita."""
    truth = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    pred = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(truth * pred)
    return (intersection + smooth) / (
        tf.reduce_sum(truth + pred) - intersection + smooth)


def iou_loss(y_true, y_pred):
    return -iou(y_true, y_pred)


class OverlapMetric(tf.keras.metrics.Metric):
    """IoU/Dice del primer plano acumulados por píxel en toda la época."""
    def __init__(self, kind="iou", threshold=0.5, name=None, **kwargs):
        super().__init__(name=name or kind, dtype="float64", **kwargs)
        if kind not in {"iou", "dice"}:
            raise ValueError("kind debe ser iou o dice")
        self.kind = kind
        self.threshold = threshold
        self.tp = self.add_weight(name="tp", initializer="zeros", dtype="float64")
        self.fp = self.add_weight(name="fp", initializer="zeros", dtype="float64")
        self.fn = self.add_weight(name="fn", initializer="zeros", dtype="float64")

    def update_state(self, y_true, y_pred, sample_weight=None):
        if sample_weight is not None:
            raise ValueError("Este experimento no utiliza sample_weight por píxel.")
        truth = tf.cast(y_true > 0.5, tf.float64)
        pred = tf.cast(y_pred >= self.threshold, tf.float64)
        self.tp.assign_add(tf.reduce_sum(truth * pred))
        self.fp.assign_add(tf.reduce_sum((1 - truth) * pred))
        self.fn.assign_add(tf.reduce_sum(truth * (1 - pred)))

    def result(self):
        numerator = self.tp if self.kind == "iou" else 2 * self.tp
        denominator = numerator + self.fp + self.fn
        return tf.where(denominator > 0, numerator / tf.maximum(denominator, 1), 1.)

    def reset_state(self):
        for variable in self.variables:
            variable.assign(0.)

    def get_config(self):
        config = super().get_config()
        config.pop("dtype", None)
        return {**config, "kind": self.kind, "threshold": self.threshold}

**Métricas y función de pérdida.** Para seleccionar y comparar se emplean máscaras binarias
con umbral fijo 0,5. Sobre todos los píxeles del conjunto, $IoU=TP/(TP+FP+FN)$ y
$Dice=2TP/(2TP+FP+FN)$. No se promedian ratios por lote ni se incluye el fondo como
clase adicional en el promedio. Se comprueba $Dice=2IoU/(1+IoU)$.

Las funciones originales con `smooth=100` se conservan y se reportan aparte como métricas
**suaves de la plantilla**, acumulando una sola vez sobre todo el conjunto. No se confunden
con la evaluación binaria. Para entrenar se combina BCE con Dice suave por imagen:
la primera penaliza errores de píxel y el segundo favorece el solapamiento de la lesión.
También se informan métricas por imagen, imágenes con lesión y por paciente, porque
asignar 1 a dos máscaras vacías puede elevar los promedios por imagen.

#### Data generator

In [ ]:
def normalizer(img, mask):
    """Escala TIFF uint8, mantiene binarias las máscaras {0,1} o {0,255}."""
    img = img.astype(np.float32) / 255.0
    mask = (mask > 0).astype(np.float32)
    return img, mask


def load_pair(image_path, mask_path, image_size=None):
    image_size = IMG_SIZE if image_size is None else image_size
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image is None or mask is None:
        raise ValueError(f"Par ilegible: {image_path} / {mask_path}")
    if image.shape[:2] != mask.shape:
        raise ValueError(f"Imagen y máscara con dimensiones distintas: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    image, mask = normalizer(image, mask)
    return image, mask[..., None]


def augment_pair(image, mask):
    # Aplicar la misma transformación espacial a imagen y máscara.
    joined = tf.concat([image, mask], axis=-1)
    joined = tf.image.random_flip_left_right(joined, seed=SEED)
    joined = tf.image.random_flip_up_down(joined, seed=SEED + 1)
    rotations = tf.random.uniform([], 0, 4, dtype=tf.int32, seed=SEED + 2)
    joined = tf.image.rot90(joined, rotations)
    return joined[..., :3], joined[..., 3:]


def image_generator(data_frame, batch_size, training=False, image_size=None):
    """Dataset finito. Incluye el último lote y no mezcla orden en val/test."""
    image_size = IMG_SIZE if image_size is None else image_size
    def read_numpy(image_path, mask_path):
        return load_pair(image_path.decode(), mask_path.decode(), image_size)

    def read_tensor(image_path, mask_path):
        image, mask = tf.numpy_function(read_numpy, [image_path, mask_path],
                                       [tf.float32, tf.float32])
        image.set_shape((image_size, image_size, 3))
        mask.set_shape((image_size, image_size, 1))
        return image, mask

    ds = tf.data.Dataset.from_tensor_slices((
        data_frame.image_path.to_numpy(dtype=str),
        data_frame.mask_path.to_numpy(dtype=str)))
    if training:
        ds = ds.shuffle(len(data_frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(read_tensor, num_parallel_calls=2, deterministic=True)
    if training:
        ds = ds.map(augment_pair, num_parallel_calls=1, deterministic=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 2
    return ds.batch(batch_size, drop_remainder=False).with_options(options).prefetch(1)

## Problem: Brain MRI Segmentation

__Target__: Mask detection

<center>
    <img src=https://proyecto-grupo-1-segmentacion-de-tumores-cerebra-07173d18d3f274.pages.fing.edu.uy/assets/img/img6.png width=800>
</center>

## Dataset

Source: [Brain MRI Segmentation](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation) (Kaggle) <br>
Alternative source: [MRI_Images.zip](https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/acholo_alumnos_uai_cl/ESTVM4NmZD5HiFKWXlMzNs4B2ADtDtCcXj4Fqw4v_0gCVA?e=ThLZ8q&download=1)

La tarea es una segmentación binaria de la anormalidad anotada en MRI. Los tres canales
del TIFF se conservan en su orden de almacenamiento al leerlos con OpenCV. Se mantiene
`IMG_SIZE = 512` de la plantilla, aunque redimensionar no agrega información a la imagen
nativa. Las máscaras usan interpolación de vecino más cercano para no inventar etiquetas.
La fuente se registra junto al experimento.

In [ ]:
DATASET_URL = "https://www.kaggle.com/api/v1/datasets/download/mateuszbuda/lgg-mri-segmentation"
DATASET_PAGE = "https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation"


def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            target = (destination / info.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Ruta no válida en ZIP")
        z.extractall(destination)


if DATA_DIR_OVERRIDE is not None:
    DATA_ROOT = Path(DATA_DIR_OVERRIDE).expanduser().resolve()
else:
    DATA_ROOT = Path.cwd() / "MRI_Images"
    if not any(DATA_ROOT.rglob("*_mask.tif")):
        archive = Path.cwd() / "MRI_Images.zip"
        if not archive.exists() or not zipfile.is_zipfile(archive):
            temporary = archive.with_suffix(".zip.part")
            print("Descargando dataset de Kaggle (~749 MB)...")
            try:
                with urllib.request.urlopen(DATASET_URL, timeout=90) as response:
                    with temporary.open("wb") as output:
                        shutil.copyfileobj(response, output, 1024 * 1024)
                if not zipfile.is_zipfile(temporary):
                    raise ValueError("La descarga no es un ZIP de imágenes")
                temporary.replace(archive)
            except Exception as exc:
                raise RuntimeError(
                    "No fue posible descargar el ZIP. Descarga MRI_Images.zip desde "
                    "la fuente del enunciado, súbelo a Colab y vuelve a ejecutar esta celda, "
                    "o indica DATA_DIR_OVERRIDE con una carpeta ya extraída."
                ) from exc
        safe_extract(archive, DATA_ROOT)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(DATA_ROOT)
print("Datos:", DATA_ROOT)

In [ ]:
def build_manifest(root):
    """Empareja por nombre exacto y detecta la copia duplicada kaggle_3m/ del ZIP."""
    rows, seen = [], {}
    copied_pairs = 0
    mask_paths = sorted(Path(root).rglob("*_mask.tif"))
    if not mask_paths:
        raise ValueError(f"No se encontraron máscaras *_mask.tif en {root}")
    for mask_path in mask_paths:
        image_path = mask_path.with_name(mask_path.name.replace("_mask.tif", ".tif"))
        if not image_path.is_file():
            raise FileNotFoundError(f"Falta imagen para {mask_path}")
        patient = mask_path.parent.name
        key = (patient, image_path.name)
        image_sha = hashlib.sha256(image_path.read_bytes()).hexdigest()
        mask_sha = hashlib.sha256(mask_path.read_bytes()).hexdigest()
        if key in seen:
            if seen[key] != (image_sha, mask_sha):
                raise ValueError(f"Mismo identificador con contenido distinto: {key}")
            copied_pairs += 1
            continue
        seen[key] = (image_sha, mask_sha)
        image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if image is None or mask is None:
            raise ValueError(f"TIFF ilegible: {key}")
        if image.ndim != 3 or image.shape[-1] != 3 or mask.ndim != 2:
            raise ValueError(f"Canales inesperados: {key}")
        if image.shape[:2] != mask.shape or image.dtype != np.uint8:
            raise ValueError(f"Dimensiones o dtype inesperados: {key}")
        if not set(np.unique(mask)).issubset({0, 1, 255}):
            raise ValueError(f"Máscara no binaria: {key}")
        rows.append({"patient": patient, "slice": image_path.name,
                     "image_path": str(image_path), "mask_path": str(mask_path),
                     "image_sha256": image_sha, "mask_sha256": mask_sha,
                     "height": image.shape[0], "width": image.shape[1],
                     "positive_pixels": int(np.count_nonzero(mask)),
                     "total_pixels": int(mask.size)})
    result = pd.DataFrame(rows).sort_values(["patient", "slice"]).reset_index(drop=True)
    result["has_lesion"] = result.positive_pixels > 0
    print(f"Pares únicos: {len(result)}. Copias idénticas descartadas: {copied_pairs}.")
    return result


df = build_manifest(DATA_ROOT)
raw_files = df.image_path.tolist()
mask_files = df.mask_path.tolist()
for image_name, mask_name in zip(raw_files[:5], mask_files[:5]):
    print(Path(image_name).name, "--", Path(mask_name).name)

In [ ]:
print("Number of samples:", len(df))
print("Pacientes:", df.patient.nunique())
print("Resoluciones nativas:", df[["height", "width"]].drop_duplicates().values.tolist())
print(f"Cortes con lesión: {df.has_lesion.mean():.1%}")
print(f"Píxeles anotados: {df.positive_pixels.sum() / df.total_pixels.sum():.2%}")
display(df[["patient", "slice", "has_lesion", "positive_pixels"]].head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df.has_lesion.value_counts().reindex([False, True], fill_value=0).plot.bar(
    ax=axes[0], color=["#829AB1", "#F28C28"], rot=0)
axes[0].set(xticklabels=["Sin lesión", "Con lesión"], ylabel="Cortes MRI")
df.groupby("patient").size().hist(ax=axes[1], bins=15, color="#2B8C82")
axes[1].set(xlabel="Cortes por paciente", ylabel="Pacientes")
fig.tight_layout()
plt.show()
plt.close(fig)

#### Pre-procesamiento

In [ ]:
def split_by_patient(data_frame, seed=84):
    patients = np.sort(data_frame.patient.unique())
    trainval_ids, test_ids = train_test_split(patients, test_size=0.1, random_state=seed)
    train_ids, val_ids = train_test_split(trainval_ids, test_size=0.1, random_state=seed)
    parts = [data_frame[data_frame.patient.isin(ids)].copy().reset_index(drop=True)
             for ids in (train_ids, val_ids, test_ids)]
    for i in range(3):
        for j in range(i + 1, 3):
            assert set(parts[i].patient).isdisjoint(parts[j].patient)
            if set(parts[i].image_sha256) & set(parts[j].image_sha256):
                raise ValueError("Imágenes idénticas en particiones distintas. Revisar duplicados.")
    assert sum(map(len, parts)) == len(data_frame)
    assert set(pd.concat(parts).image_path) == set(data_frame.image_path)
    for part in parts:
        if not part.has_lesion.any():
            raise ValueError("Una partición no tiene lesiones. Revisar el dataset antes de entrenar.")
    return parts


df_train, df_val, df_test = split_by_patient(df, SEED)
split_table = pd.DataFrame([
    {"partición": name, "pacientes": part.patient.nunique(), "imágenes": len(part),
     "% imágenes": 100 * len(part) / len(df),
     "% con lesión": 100 * part.has_lesion.mean()}
    for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]
])
display(split_table.round(2))
manifest = pd.concat([part.assign(split=name) for name, part in
                      [("train", df_train), ("val", df_val), ("test", df_test)]])
signature_columns = ["patient", "slice", "image_sha256", "mask_sha256", "split"]
SPLIT_SIGNATURE = hashlib.sha256(
    manifest[signature_columns].sort_values(["patient", "slice"]).to_csv(index=False).encode()
).hexdigest()
EXPERIMENT_ROOT = OUTPUT_ROOT / SPLIT_SIGNATURE[:12]
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
manifest.to_csv(EXPERIMENT_ROOT / "particiones.csv", index=False)
(EXPERIMENT_ROOT / "entorno.json").write_text(json.dumps(ENVIRONMENT, indent=2))
print("Huella de datos y particiones:", SPLIT_SIGNATURE)

**Protocolo común.** Se mantiene la proporción de la plantilla: 10% de pacientes para test y
10% del remanente para validación (aproximadamente 81/9/10 en pacientes, no necesariamente
en imágenes). Se corrige la partición por corte para evitar información del mismo paciente
en conjuntos diferentes. El test queda reservado hasta fijar el modelo en la pregunta 3.

La semilla, resolución, batch, aumentos, pérdida, optimizador y presupuesto máximo de épocas
son comunes. Se conservan todos los cortes, incluyendo los negativos. Solo train recibe
reflexiones y rotaciones múltiplos de 90°, aplicadas conjuntamente a imagen y máscara.
Validación y test son deterministas. `EarlyStopping` controla `val_iou`, reduce la tasa cuando
se estanca y restaura el checkpoint de mayor IoU. Esto limita el sobreajuste, pero no garantiza
su ausencia: se examinan curvas y diferencias de train/val con los mejores pesos.

**Arquitectura.** Cada etapa tiene un codificador y un decodificador completos. Profundidad 4
significa cuatro reducciones y un fondo, con filtros `[15, 30, 60, 120, 240]` en la pregunta 1.
Los decodificadores se conectan al siguiente codificador por sumas a igual resolución.
Cada bloque residual reutiliza una misma capa convolucional dos veces, compartiendo pesos.
La salida es una máscara de un canal con `sigmoid`.

**Referencia y alcance docente.** La definición usada es una adaptación TensorFlow/Keras de
[LadderNet de Juntang Zhuang](https://github.com/juntang-zhuang/LadderNet/blob/master/src/LadderNetv65.py)
y su [artículo](https://arxiv.org/abs/1810.07810). Se mantienen las conexiones y bloques
compartidos, se parametriza el número de etapas y se usa una salida binaria.
El notebook entregado no trae la definición vista en clase y esa implementación no estuvo
disponible para contrastarla. Por ello no se afirma equivalencia exacta con el código docente.

In [ ]:
def shared_residual(x, filters, name, dropout=0.25):
    if x.shape[-1] != filters:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu",
                          name=name + "_projection")(x)
    shared_conv = layers.Conv2D(filters, 3, padding="same", name=name + "_shared")
    h = layers.Activation("relu", name=name + "_relu1")(shared_conv(x))
    h = layers.SpatialDropout2D(dropout, name=name + "_dropout")(h)
    h = shared_conv(h)  # La misma instancia: pesos compartidos de verdad.
    return layers.Activation("relu", name=name + "_relu2")(
        layers.Add(name=name + "_residual")([x, h]))


def build_laddernet(stages=2, initial_filters=15, depth=4, image_size=None,
                    dropout=None):
    # Keras 3 restablece la política al hacer clear_session: fijarla en cada modelo.
    tf.keras.mixed_precision.set_global_policy(PRECISION_POLICY)
    image_size = IMG_SIZE if image_size is None else image_size
    dropout = DROPOUT if dropout is None else dropout
    if stages < 1 or depth < 1 or initial_filters < 1 or image_size % (2 ** depth):
        raise ValueError("Configuración inválida o tamaño no divisible por 2**depth")
    inputs = layers.Input((image_size, image_size, 3), name="mri")
    h = layers.Conv2D(initial_filters, 3, padding="same", activation="relu",
                      name="input_projection")(inputs)
    previous_decoder = None
    for stage in range(stages):
        prefix = f"stage{stage + 1}"
        if previous_decoder is not None:
            h = shared_residual(previous_decoder[0], initial_filters,
                                prefix + "_input", dropout)
        skips = []
        for level in range(depth):
            width = initial_filters * 2 ** level
            if previous_decoder is not None:
                h = layers.Add(name=f"{prefix}_interstage{level}")(
                    [h, previous_decoder[level]])
            h = shared_residual(h, width, f"{prefix}_encoder{level}", dropout)
            skips.append(h)
            h = layers.Conv2D(width * 2, 3, strides=2, padding="same", activation="relu",
                              name=f"{prefix}_down{level}")(h)
        h = shared_residual(h, initial_filters * 2 ** depth, prefix + "_bottom", dropout)
        decoded = {depth: h}
        for level in reversed(range(depth)):
            width = initial_filters * 2 ** level
            h = layers.Conv2DTranspose(width, 3, strides=2, padding="same",
                                       name=f"{prefix}_up{level}")(h)
            h = layers.Add(name=f"{prefix}_skip{level}")([h, skips[level]])
            h = shared_residual(h, width, f"{prefix}_decoder{level}", dropout)
            decoded[level] = h
        previous_decoder = decoded
    outputs = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32",
                            name="mask_probability")(h)
    return Model(inputs, outputs, name=f"LadderNet_{stages}stages_f{initial_filters}_d{depth}")


def compile_model(model):
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE, clipnorm=1.0),
                  loss=segmentation_loss,
                  metrics=[OverlapMetric("iou", THRESHOLD), OverlapMetric("dice", THRESHOLD)],
                  jit_compile=False)
    return model


def overlap_from_counts(tp, fp, fn):
    tp, fp, fn = np.asarray(tp, dtype=float), np.asarray(fp, dtype=float), np.asarray(fn, dtype=float)
    union = tp + fp + fn
    denom = 2 * tp + fp + fn
    ious = np.divide(tp, union, out=np.ones_like(union), where=union != 0)
    dices = np.divide(2 * tp, denom, out=np.ones_like(denom), where=denom != 0)
    return ious, dices


def evaluate_frame(model, frame):
    records, offset = [], 0
    soft_intersection = soft_truth = soft_pred = 0.0
    for images, masks in image_generator(frame, BATCH_SIZE, training=False):
        probability = model(images, training=False).numpy()
        if not np.isfinite(probability).all():
            raise FloatingPointError("Predicciones no finitas")
        assert np.min(probability) >= 0 and np.max(probability) <= 1
        truth = masks.numpy() > 0.5
        prediction = probability >= THRESHOLD
        soft_intersection += np.sum(truth * probability, dtype=np.float64)
        soft_truth += np.sum(truth, dtype=np.float64)
        soft_pred += np.sum(probability, dtype=np.float64)
        for index in range(len(probability)):
            row = frame.iloc[offset + index]
            tp = int(np.count_nonzero(truth[index] & prediction[index]))
            fp = int(np.count_nonzero(~truth[index] & prediction[index]))
            fn = int(np.count_nonzero(truth[index] & ~prediction[index]))
            binary_iou, binary_dice = overlap_from_counts(tp, fp, fn)
            records.append({"patient": row.patient, "slice": row.slice,
                            "tp": tp, "fp": fp, "fn": fn,
                            "has_lesion": bool(truth[index].any()),
                            "iou": float(binary_iou), "dice": float(binary_dice)})
        offset += len(probability)
    assert offset == len(frame), "No se evaluaron todas las imágenes"
    details = pd.DataFrame(records)
    totals = details[["tp", "fp", "fn"]].sum()
    global_iou, global_dice = overlap_from_counts(*totals)
    patient_counts = details.groupby("patient")[["tp", "fp", "fn"]].sum()
    patient_iou, patient_dice = overlap_from_counts(
        patient_counts.tp, patient_counts.fp, patient_counts.fn)
    positive = details[details.has_lesion]
    report = {
        "iou": float(global_iou), "dice": float(global_dice), "n_images": len(frame),
        "n_patients": int(frame.patient.nunique()),
        "iou_macro_images": float(details.iou.mean()),
        "dice_macro_images": float(details.dice.mean()),
        "iou_positive_images": float(positive.iou.mean()),
        "dice_positive_images": float(positive.dice.mean()),
        "iou_macro_patients": float(np.mean(patient_iou)),
        "dice_macro_patients": float(np.mean(patient_dice)),
        "soft_iou_template": float((soft_intersection + 100) / (
            soft_truth + soft_pred - soft_intersection + 100)),
        "soft_dice_template": float((2 * soft_intersection + 100) / (
            soft_truth + soft_pred + 100)),
    }
    np.testing.assert_allclose(report["dice"], 2 * report["iou"] / (1 + report["iou"]), atol=1e-7)
    return report, details


class FiniteLogs(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if any(not np.isfinite(float(value)) for value in (logs or {}).values()):
            raise FloatingPointError(f"Métrica no finita en época {epoch + 1}")


def restore_model(run):
    model = build_laddernet(**run["architecture"])
    model.load_weights(Path(run["run_dir"]) / "best.weights.h5")
    return model


def run_experiment(label, architecture):
    config = {"schema": "tarea3-v1", "architecture": architecture, "seed": SEED,
              "image_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs": MAX_EPOCHS,
              "patience": PATIENCE, "lr": LEARNING_RATE, "dropout": DROPOUT,
              "threshold": THRESHOLD, "loss": "0.5BCE+0.5Dice",
              "augmentation": "flips+rot90", "split": SPLIT_SIGNATURE,
              "tensorflow": tf.__version__, "keras": tf.keras.__version__,
              "precision": PRECISION_POLICY}
    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
    run_dir = EXPERIMENT_ROOT / (label + "_" + fingerprint)
    run_dir.mkdir(parents=True, exist_ok=True)
    complete = run_dir / "resultado.json"
    if REUSE_COMPLETED and complete.exists():
        result = json.loads(complete.read_text())
        if result["config"] == config and (run_dir / "best.weights.h5").exists():
            result["run_dir"] = str(run_dir)
            print("Reutilizando experimento completo:", label)
            return result
    complete.unlink(missing_ok=True)
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    model = compile_model(build_laddernet(**architecture))
    print(f"{label}: {architecture}, parámetros={model.count_params():,}")
    (run_dir / "config.json").write_text(json.dumps(config, indent=2))
    (run_dir / "architecture.json").write_text(model.to_json())
    callbacks = [
        FiniteLogs(),
        tf.keras.callbacks.ModelCheckpoint(str(run_dir / "best.weights.h5"),
            monitor="val_iou", mode="max", save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_iou", mode="max", factor=0.5,
            patience=4, min_delta=0., min_lr=1e-6, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_iou", mode="max", patience=PATIENCE,
            min_delta=0., restore_best_weights=True, verbose=1),
        tf.keras.callbacks.CSVLogger(str(run_dir / "history.csv")),
    ]
    start = time.perf_counter()
    history_obj = model.fit(
        image_generator(df_train, BATCH_SIZE, training=True),
        validation_data=image_generator(df_val, BATCH_SIZE, training=False),
        epochs=MAX_EPOCHS, callbacks=callbacks, verbose=2)
    duration = time.perf_counter() - start
    history = {key: [float(x) for x in values] for key, values in history_obj.history.items()}
    # Recargar el archivo asegura evaluar exactamente la época guardada.
    model.load_weights(run_dir / "best.weights.h5")
    val, val_details = evaluate_frame(model, df_val)
    train, _ = evaluate_frame(model, df_train)  # Sin aumento ni dropout, comparables con val.
    np.testing.assert_allclose(val["iou"], max(history["val_iou"]), atol=1e-6)
    val_details.to_csv(run_dir / "validacion_por_imagen.csv", index=False)
    result = {"label": label, "architecture": architecture, "config": config,
              "run_dir": str(run_dir), "params": int(model.count_params()),
              "epochs_run": len(history["loss"]),
              "best_epoch": int(np.argmax(history["val_iou"]) + 1),
              "training_seconds": duration, "history": history, "val": val, "train": train}
    (run_dir / "resultado.tmp.json").write_text(json.dumps(result, indent=2, allow_nan=False))
    (run_dir / "resultado.tmp.json").replace(complete)
    del model, history_obj
    tf.keras.backend.clear_session()
    gc.collect()
    return result


def report_run(run):
    display(pd.DataFrame({"validación": run["val"]}).round(5))
    display(Markdown(
        f"**{run['label']}** — IoU de validación **{run['val']['iou']:.4f}**, "
        f"Dice **{run['val']['dice']:.4f}**. Mejor época: **{run['best_epoch']}** de "
        f"{run['epochs_run']} ejecutadas. Parámetros: {run['params']:,}. "
        f"Entrenamiento: {run['training_seconds'] / 60:.1f} minutos."))
    plot_history(run["history"], save_path=Path(run["run_dir"]) / "curvas.png")


# Validaciones previas: formas, rangos, máscara binaria y enlace entre etapas.
example_images, example_masks = next(iter(image_generator(df_train, BATCH_SIZE)))
assert example_images.shape[1:] == (IMG_SIZE, IMG_SIZE, 3)
assert example_masks.shape[1:] == (IMG_SIZE, IMG_SIZE, 1)
assert set(np.unique(example_masks)).issubset({0., 1.})
assert np.isfinite(example_images).all() and float(tf.reduce_max(example_images)) <= 1
print("Preprocesamiento validado. Las preguntas siguientes entrenan los modelos.")

## Pregunta 1 (5 pts):

Usando el código vista en clase, entrene un modelo LadderNet de 2 etapas con 15 filtros iniciales, una profundida de 4 y una cantidad de épocas que evite sobreajuste. Entregue el valor de la métrica IoU y DICE del conjunto de validación.  

Se fijan exactamente **2 etapas, 15 filtros iniciales y profundidad 4**. Se entrena desde
cero y se conserva la época con mayor IoU de validación. El máximo de 80 épocas permite
aprender y la parada temprana evita continuar cuando la validación deja de mejorar.
Las métricas reportadas corresponden a los mejores pesos, no necesariamente a la última época.

In [ ]:
Q1_ARCHITECTURE = {"stages": 2, "initial_filters": 15, "depth": 4}
q1_result = run_experiment("P1_LadderNet_2_etapas", Q1_ARCHITECTURE)

In [ ]:
report_run(q1_result)

## Pregunta 2 (5 pts):

Usando el código vista en clase, entrene un modelo LadderNet de 3 etapas con una cantidad de filtros iniciales y profundidad que usted elijan y una cantidad de épocas que evite sobreajuste con el objetivo que puedan maximizar el IoU en el conjunto de validación. Entregue el valor de la métrica IoU y DICE del conjunto de validación.  

Se comparan tres configuraciones predefinidas de **3 etapas**:

| Filtros iniciales | Profundidad | Motivo |
|---:|---:|---|
| 15 | 4 | Aislar el efecto de agregar una etapa respecto de P1 |
| 24 | 4 | Aumentar la capacidad de representación en cada nivel |
| 16 | 5 | Incorporar una reducción adicional para ampliar el contexto |

Cada candidato recibe el mismo presupuesto máximo que P1. Se selecciona por **mayor IoU
global de validación**, desempate por menos parámetros. La búsqueda es acotada y no garantiza
un óptimo global. Su costo total se muestra en la comparación. Test no participa.

In [ ]:
q2_candidates = []
for architecture in Q2_CONFIGS:
    label = f"P2_LadderNet_3_f{architecture['initial_filters']}_d{architecture['depth']}"
    q2_candidates.append(run_experiment(label, architecture))
q2_result = sorted(q2_candidates, key=lambda r: (-r["val"]["iou"], r["params"]))[0]
search_table = pd.DataFrame([
    {"candidato": r["label"], **r["architecture"], "IoU val": r["val"]["iou"],
     "Dice val": r["val"]["dice"], "época mejor": r["best_epoch"],
     "épocas ejecutadas": r["epochs_run"], "parámetros": r["params"],
     "minutos": r["training_seconds"] / 60}
    for r in q2_candidates]).sort_values("IoU val", ascending=False)
search_table.to_csv(EXPERIMENT_ROOT / "busqueda_p2.csv", index=False)
display(search_table.round(4))
print("Candidato seleccionado:", q2_result["label"])

In [ ]:
report_run(q2_result)

## Pregunta 3 (2 pts):

Analicen y comparen los resultados obtenidos en las preguntas 1 y 2. Indique ¿Cuál de los dos modelos se quedarían? justifiquen. 

La elección principal se basa en IoU de validación. Se considera empate práctico una diferencia
menor o igual a **0,005** (0,5 puntos porcentuales), fijada antes de observar resultados.
En ese caso se prefiere el modelo con menos parámetros. Se comparan también Dice, generalización,
costo de entrenamiento y variabilidad entre pacientes. IoU y Dice globales son transformaciones
monótonas entre sí, por lo que su coincidencia no constituye dos evidencias independientes.

In [ ]:
comparison = pd.DataFrame([
    {"modelo": r["label"], "IoU validación": r["val"]["iou"],
     "Dice validación": r["val"]["dice"], "IoU train sin aumento": r["train"]["iou"],
     "brecha train-val": r["train"]["iou"] - r["val"]["iou"],
     "Dice cortes con lesión": r["val"]["dice_positive_images"],
     "IoU promedio pacientes": r["val"]["iou_macro_patients"],
     "parámetros": r["params"], "mejor época": r["best_epoch"],
     "épocas": r["epochs_run"], "minutos": r["training_seconds"] / 60}
    for r in (q1_result, q2_result)]).set_index("modelo")
display(comparison.round(4))
comparison.to_csv(EXPERIMENT_ROOT / "comparacion_p1_p2.csv")
print(f"Costo total de búsqueda P2: {sum(r['training_seconds'] for r in q2_candidates)/60:.1f} minutos")


def paired_bootstrap_iou(run1, run2, repeats=2000, seed=84):
    a = pd.read_csv(Path(run1["run_dir"]) / "validacion_por_imagen.csv")
    b = pd.read_csv(Path(run2["run_dir"]) / "validacion_por_imagen.csv")
    a = a.groupby("patient")[["tp", "fp", "fn"]].sum().sort_index()
    b = b.groupby("patient")[["tp", "fp", "fn"]].sum().reindex(a.index)
    if b.isna().any().any():
        raise ValueError("Pacientes no comparables")
    sample = np.random.default_rng(seed).integers(0, len(a), size=(repeats, len(a)))
    ca, cb = a.to_numpy()[sample].sum(axis=1), b.to_numpy()[sample].sum(axis=1)
    ia, _ = overlap_from_counts(ca[:, 0], ca[:, 1], ca[:, 2])
    ib, _ = overlap_from_counts(cb[:, 0], cb[:, 1], cb[:, 2])
    return np.quantile(ib - ia, [0.025, 0.975])


ci_low, ci_high = paired_bootstrap_iou(q1_result, q2_result)
print(f"IC bootstrap descriptivo 95% de IoU(P2)-IoU(P1): [{ci_low:.4f}, {ci_high:.4f}]")

In [ ]:
IOU_TOLERANCE = 0.005
difference = q2_result["val"]["iou"] - q1_result["val"]["iou"]
if abs(difference) <= IOU_TOLERANCE:
    selected_result = min([q1_result, q2_result], key=lambda r: r["params"])
    reason = ("la diferencia cae dentro del empate práctico predefinido de 0,005 "
              "y se prefiere la menor cantidad de parámetros")
else:
    selected_result = max([q1_result, q2_result], key=lambda r: r["val"]["iou"])
    reason = "presenta mayor IoU de validación fuera de la tolerancia de empate"
gap = selected_result["train"]["iou"] - selected_result["val"]["iou"]
uncertainty = ("El intervalo incluye cero, por lo que la ventaja no es concluyente entre pacientes."
               if ci_low <= 0 <= ci_high else
               "El intervalo descriptivo no incluye cero en esta partición de pacientes.")
generalization = ("La brecha positiva es compatible con sobreajuste o diferencias entre pacientes."
                  if gap > 0.05 else
                  "La brecha observada es acotada, aunque no prueba ausencia de sobreajuste."
                  if gap >= 0 else
                  "Validación supera a train, lo que puede reflejar diferencias entre pacientes.")
if selected_result["val"]["iou"] < 0.05:
    generalization = ("El solapamiento es muy bajo: el modelo aún no ofrece una segmentación "
                      "útil. Una brecha pequeña o un empate no implican buen desempeño. "
                      "Hay que revisar el entrenamiento y las predicciones antes de entregar.")
text = (
    f"### Respuesta y elección del modelo\n\n"
    f"P1 obtiene IoU **{q1_result['val']['iou']:.4f}** y Dice **{q1_result['val']['dice']:.4f}**. "
    f"El mejor P2 obtiene IoU **{q2_result['val']['iou']:.4f}** y Dice **{q2_result['val']['dice']:.4f}**. "
    f"La diferencia de IoU P2 − P1 es **{difference:.4f}** ({difference*100:.2f} puntos porcentuales).\n\n"
    f"**Nos quedaríamos con {selected_result['label']}**, porque {reason}. "
    f"Tiene {selected_result['params']:,} parámetros, su mejor época es "
    f"{selected_result['best_epoch']} y la brecha IoU train − val es {gap:.4f}. "
    f"{generalization} La curva completa permite revisar cuándo dejó de mejorar.\n\n"
    f"{uncertainty} El bootstrap es pareado por paciente y mantiene unidos sus cortes. "
    f"Es descriptivo: la selección de hiperparámetros se hizo sobre esta misma validación y "
    f"el intervalo no corrige ese sesgo. Más etapas no garantizan mejor generalización. "
    f"La comparación depende de una semilla y de esta partición. Test se utiliza a continuación "
    f"para evaluar una sola vez el modelo ya elegido."
)
display(Markdown(text))
(EXPERIMENT_ROOT / "respuesta_p3.md").write_text(text, encoding="utf-8")
(EXPERIMENT_ROOT / "modelo_seleccionado.json").write_text(json.dumps({
    "label": selected_result["label"], "architecture": selected_result["architecture"],
    "run_dir": selected_result["run_dir"], "threshold": THRESHOLD,
    "reason": reason, "validation_iou": selected_result["val"]["iou"],
    "iou_tolerance": IOU_TOLERANCE, "bootstrap_difference_ci": [float(ci_low), float(ci_high)]
}, indent=2))

## Pregunta 4 (3 pts):

Del modelo seleccionado en la pregunta 3, muestre 10 predicciones del conjunto de test con su respectivas imagenes reales y de MRI's.   

Se utiliza exclusivamente el modelo fijado en P3 y su umbral 0,5. Se muestran **10 cortes de test**
seleccionados con semilla fija antes de predecir: hasta 7 con lesión y el resto sin lesión.
Esta selección visual permite revisar falsos positivos y bordes, y no se usa como estimación
del rendimiento. Las métricas de test se calculan sobre **todo** el conjunto.
Las columnas muestran MRI, máscara real, máscara predicha y superposición de contornos.

In [ ]:
assert len(df_test) >= 10, "Se necesitan al menos diez imágenes en test"
tf.keras.backend.clear_session()
selected_model = restore_model(selected_result)

# Selección reproducible basada en la anotación, no en la calidad de predicción.
positive_pool = df_test[df_test.has_lesion]
negative_pool = df_test[~df_test.has_lesion]
n_positive = min(7, len(positive_pool))
n_negative = min(10 - n_positive, len(negative_pool))
chosen = pd.concat([positive_pool.sample(n_positive, random_state=SEED),
                    negative_pool.sample(n_negative, random_state=SEED)])
if len(chosen) < 10:
    remaining = df_test[~df_test.image_path.isin(chosen.image_path)]
    chosen = pd.concat([chosen, remaining.sample(10 - len(chosen), random_state=SEED)])
chosen = chosen.reset_index(drop=True)
chosen[["patient", "slice", "has_lesion"]].to_csv(EXPERIMENT_ROOT / "seleccion_visual_test.csv", index=False)

test_metrics, test_details = evaluate_frame(selected_model, df_test)
display(pd.DataFrame({"test completo": test_metrics}).round(5))
test_details.to_csv(EXPERIMENT_ROOT / "test_por_imagen.csv", index=False)
(EXPERIMENT_ROOT / "metricas_test.json").write_text(json.dumps(test_metrics, indent=2, allow_nan=False))

fig, axes = plt.subplots(10, 4, figsize=(14, 30))
for row_index, row in chosen.iterrows():
    image, real_mask = load_pair(row.image_path, row.mask_path)
    probability = selected_model(image[None], training=False).numpy()[0, ..., 0]
    prediction = probability >= THRESHOLD
    mask = real_mask[..., 0] > 0.5
    values = test_details[(test_details.patient == row.patient) & (test_details.slice == row.slice)].iloc[0]
    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(f"{row.patient}\n{row.slice}", fontsize=7)
    axes[row_index, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[row_index, 1].set_title("Máscara real", fontsize=9)
    axes[row_index, 2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    axes[row_index, 2].set_title(f"Predicción | IoU {values.iou:.3f} | Dice {values.dice:.3f}", fontsize=8)
    axes[row_index, 3].imshow(image)
    if mask.any() and not mask.all():
        axes[row_index, 3].contour(mask, levels=[0.5], colors=["cyan"], linewidths=0.8)
    if prediction.any() and not prediction.all():
        axes[row_index, 3].contour(prediction, levels=[0.5], colors=["orange"], linewidths=0.8)
    axes[row_index, 3].set_title("Real: cian | Predicción: naranja", fontsize=8)
    for ax in axes[row_index]:
        ax.axis("off")
fig.tight_layout()
fig.savefig(EXPERIMENT_ROOT / "10_predicciones_test.png", dpi=130, bbox_inches="tight")
plt.show()
plt.close(fig)

assert test_metrics["n_images"] == len(df_test)
assert len(chosen) == 10 and chosen.image_path.nunique() == 10
assert set(chosen.image_path).issubset(set(df_test.image_path))
assert q1_result["architecture"] == {"stages": 2, "initial_filters": 15, "depth": 4}
assert q2_result["architecture"]["stages"] == 3
for result in [q1_result, *q2_candidates]:
    assert all(np.isfinite(v) for key, v in result["val"].items())
    assert 0 <= result["val"]["iou"] <= 1 and 0 <= result["val"]["dice"] <= 1
    assert (Path(result["run_dir"]) / "best.weights.h5").is_file()
display(Markdown(
    f"**P4:** se muestran las 10 predicciones requeridas. En todo test, el modelo seleccionado "
    f"obtiene IoU **{test_metrics['iou']:.4f}** y Dice **{test_metrics['dice']:.4f}**, "
    f"sobre {test_metrics['n_images']} imágenes de {test_metrics['n_patients']} pacientes. "
    "La inspección visual permite identificar omisiones de lesiones, sobresegmentación y "
    "falsos positivos en cortes negativos. Estos resultados describen esta partición y "
    "no constituyen una validación clínica.\n\n"
    "**Ejecución terminada.** Guardar el notebook con sus salidas antes de enviarlo a Webcursos."))
print("Archivos del experimento:", EXPERIMENT_ROOT)

**Referencias**

- Zhuang, J. *LadderNet: Multi-path networks based on U-Net for medical image segmentation*.
  [Artículo](https://arxiv.org/abs/1810.07810) y [código original](https://github.com/juntang-zhuang/LadderNet).
- Buda, M. [Brain MRI segmentation — LGG dataset](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation).
- TensorFlow. [EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping)
  y [ModelCheckpoint](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ModelCheckpoint).
- Keras. [Métricas de segmentación y acumulación de IoU](https://keras.io/api/metrics/segmentation_metrics/).